In [1]:
import pandas as pd
from torch.utils.data.dataset import _T_co

TRAIN_URL = (
    "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
)

df = pd.read_csv(
    TRAIN_URL,
    sep="\t"
)

In [2]:
df = df.dropna(subset=["document"])
df.shape[0]

149995

In [3]:
df = df.sample(
    n=12000,
    random_state=42
).reset_index(drop=True)

In [4]:
df.head()

,id,document,label
0,7865795,원본이 최고,1
1,5417631,스릴감과 훈훈함이 있는 영화.,1
2,8357466,굉장히 저평가되는 영화중 하나라고 생각함,1
3,8252946,정말영화같은이야기 영화여서 영화같은이야기가 좋다,1
4,7800452,계기도없는데 이상하다,0


In [16]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=2000,
    random_state=42,
    stratify=df["label"]
)

train_df.head()

,id,document,label
2327,9895115,다양한 생각을 자연스럽게 인정하는 일본 그래서 문학이 발전하고 기발한 애니매이션도 ...,1
3323,4283815,전체적인 분위기가 너무 좋은. 결말이 황당.,1
510,3055710,장률의 또 하나의 걸작!,1
974,10246514,0점각 m창인생하앙더떄려줘똘킹탑간다,0
6049,9080687,시간때우기로는 그냥...근데 여주인공 너무 삭았음..아줌마인줄알았는데,0


In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [9]:
text = "이 영화 정말 재미있다"
encoded = tokenizer(text)
encoded

{'input_ids': [2, 1504, 3771, 3944, 6001, 2062, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [11]:
tokenizer.convert_ids_to_tokens(encoded["input_ids"])

['[CLS]', '이', '영화', '정말', '재미있', '##다', '[SEP]']

In [13]:
import torch
from torch.utils.data import Dataset

class NSMCDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df["document"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=128,
        )

        encoded["labels"] = label

        return encoded

In [18]:
train_dataset = NSMCDataset(
    train_df,
    tokenizer
)

val_dataset = NSMCDataset(
    val_df,
    tokenizer
)


In [19]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [20]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collator
)

In [22]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape)

input_ids torch.Size([16, 83])
token_type_ids torch.Size([16, 83])
attention_mask torch.Size([16, 83])
labels torch.Size([16])


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2 # pos와 neg 2가지 케이스가 존재함.
)

In [24]:
outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    token_type_ids=batch["token_type_ids"],
    labels=batch["labels"]
)

In [28]:
outputs.logits.shape, outputs.logits.argmax(dim=1), outputs.logits

(torch.Size([16, 2]),
 tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]),
 tensor([[ 0.3000, -0.1611],
         [ 0.3521,  0.2068],
         [ 0.2619, -0.3404],
         [ 0.2843, -0.1374],
         [ 0.2223, -0.0940],
         [ 0.0352,  0.2516],
         [ 0.5052,  0.1888],
         [ 0.3686,  0.1535],
         [ 0.4016,  0.0661],
         [ 0.5664, -0.1415],
         [ 0.1067,  0.1193],
         [ 0.7427,  0.1688],
         [ 0.6275, -0.1322],
         [ 0.4387,  0.0913],
         [ 0.3806,  0.1364],
         [ 0.4972,  0.4319]], grad_fn=<AddmmBackward0>))

In [29]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(DEVICE)

In [30]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

In [31]:
for epoch in range(3):
    model.train()

    total_loss = 0

    for batch in train_loader:

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        # input_ids, attention_mask, token_type_ids, labels
        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    train_loss = (
        total_loss / len(train_loader)
    )

    print(
        f"epoch: {epoch+1} | train_loss: {train_loss:.4f}"
    )

epoch: 1 | train_loss: 0.3579
epoch: 2 | train_loss: 0.2087
epoch: 3 | train_loss: 0.1124


In [32]:
from sklearn.metrics import accuracy_score, f1_score

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        logits = outputs.logits

        preds = logits.argmax(dim=-1)

        all_preds.extend(
            preds.cpu().tolist()
        )

        all_labels.extend(
            batch["labels"].cpu().tolist()
        )

In [34]:
accuracy = accuracy_score(
    all_labels,
    all_preds
)

f1 = f1_score(
    all_labels,
    all_preds
)

print(f"accuracy: {accuracyb}")
print(f"f1: {f1b}")

accuracy: 0.884
f1: 0.8825910931174089


In [43]:
text = "굉장히 기분나쁘지만 그래도 다시 볼 가치가 있다. 기분이 안좋았다. 절대 보고싶지 않다. 그래도 볼 것이다. 절대 안봐"

inputs = tokenizer(
    text,
    return_tensors="pt"
)

inputs = {
    k: v.to(DEVICE)
    for k, v in inputs.items()
}

model.eval()

with torch.no_grad():

    outputs = model(**inputs)

    print(torch.softmax(outputs.logits, dim=1))

    pred = outputs.logits.argmax(
        dim=-1
    )

print(pred.item())

tensor([[0.7419, 0.2581]], device='cuda:0')
0
